# AL Experiments

Imports `core/` directly — no server, no Supabase connection required.

Loads mock data from the existing SQLite DB created by `2_uroflow_labeling/active_learning/uro_active_learning.ipynb`. If that DB does not exist, run the notebook there once to generate it.

Use this notebook to compare query strategies, inspect score distributions, and sketch learning curves — without touching the FastAPI service.

In [ ]:
import sys, os
from pathlib import Path

# Make `core` importable when the notebook runs from 4_al_backend/notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from core.config import FEATURES, LABEL_OPTIONS
from core.data import enrich_ipss_features, load_measurements_from_sqlite
from core.model import PatientModel
from core.query_strategy import UncertaintySampler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

## Load mock data

Point `DB_PATH` at the SQLite DB from `2_uroflow_labeling/active_learning/data/db/data_mock.db`.

In [ ]:
DB_PATH = '../../2_uroflow_labeling/active_learning/data/db/data_mock.db'
assert os.path.exists(DB_PATH), f'Mock DB not found at {DB_PATH} — run the uro_active_learning notebook once to generate it.'

meas, ipss = load_measurements_from_sqlite(DB_PATH)
df = enrich_ipss_features(meas, ipss)
print(f'Loaded {len(df)} measurements, {df["label"].notna().sum()} labeled')
df.head()

## Train a model on labeled data

For experimentation we split labeled data into train/pool to simulate an active-learning scenario — the pool plays the role of unlabeled measurements we would query for labels.

In [ ]:
labeled = df[df['label'].notna()].reset_index(drop=True)
X = labeled[FEATURES].astype(float)
y = labeled['label'].astype(str)

X_train, X_pool, y_train, y_pool = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42
)

model = PatientModel().fit(X_train, y_train)
print(f'Trained on {len(X_train)} samples | classes: {list(model.classes_)}')

## Compare query strategies on the same pool

All three strategies score the pool with probabilities from the *same* model — so any differences come purely from how uncertainty is measured.

In [ ]:
proba_pool = model.predict_proba(X_pool)

scores = {
    s: UncertaintySampler(s).score_from_proba(proba_pool)
    for s in ('entropy', 'margin', 'least_confident')
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, s) in zip(axes, scores.items()):
    ax.hist(s, bins=25, color='#5B8FC9', edgecolor='white', alpha=0.85)
    ax.set_title(f'{name}  (mean {s.mean():.2f})')
    ax.set_xlabel('uncertainty')
axes[0].set_ylabel('count')
fig.suptitle('Pool uncertainty — distribution per strategy', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Top-10 pool overlap between strategies — do they agree on what is hard?
TOPN = 10
tops = {s: set(np.argsort(-v)[:TOPN]) for s, v in scores.items()}
for a in tops:
    for b in tops:
        if a < b:
            overlap = len(tops[a] & tops[b])
            print(f'{a:>16} ∩ {b:<16}: {overlap}/{TOPN}')

## Learning curve

Start with a small labeled seed, iteratively query the `n` most-uncertain pool samples under each strategy, reveal their ground-truth labels, retrain, and track test accuracy. This is the classic AL evaluation protocol.

In [ ]:
def al_curve(strategy, batch_size=5, iters=10, seed=0):
    from sklearn.utils import resample
    rng = np.random.default_rng(seed)

    # Initial split: 20 seed labels, rest into pool + test
    labeled_idx = rng.choice(len(X), size=20, replace=False)
    remaining = np.setdiff1d(np.arange(len(X)), labeled_idx)
    test_idx = remaining[: len(remaining) // 2]
    pool_idx = remaining[len(remaining) // 2 :]

    X_t, y_t = X.iloc[test_idx], y.iloc[test_idx]

    accs, sizes = [], []
    sampler = UncertaintySampler(strategy)

    for _ in range(iters):
        m = PatientModel().fit(X.iloc[labeled_idx], y.iloc[labeled_idx])
        acc = (m.predict(X_t) == y_t.values).mean()
        accs.append(acc); sizes.append(len(labeled_idx))

        if len(pool_idx) == 0:
            break
        picks = sampler.query(m, X.iloc[pool_idx], n=min(batch_size, len(pool_idx)))
        chosen = pool_idx[picks]
        labeled_idx = np.concatenate([labeled_idx, chosen])
        pool_idx = np.setdiff1d(pool_idx, chosen)

    return sizes, accs

fig, ax = plt.subplots(figsize=(8, 4))
for strat in ('entropy', 'margin', 'least_confident'):
    sizes, accs = al_curve(strat)
    ax.plot(sizes, accs, '-o', label=strat, alpha=0.85)
ax.set_xlabel('labeled samples'); ax.set_ylabel('test accuracy')
ax.set_title('Learning curve per strategy', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## Switch strategy at runtime

All three are interchangeable — set the env var `AL_STRATEGY` (used by the FastAPI service) or pass `strategy=...` here.

In [ ]:
STRATEGY = 'entropy'  # 'entropy' | 'margin' | 'least_confident'
idx = UncertaintySampler(STRATEGY).query(model, X_pool, n=5)
X_pool.iloc[idx].assign(predicted=model.predict(X_pool.iloc[idx]))